# 🇹🇷 AUTSL 10 Sınıf: Sıfır El ROI + Motion2 Deney Eğitimi
Bu notebook, **22 Ekim 2026** tarihli `zero_hand_upperbody_roi_motion2` önbelleği üzerinde **Seed 7** ile 20 epochluk kontrollü model eğitimini ve resmi **Val** değerlendirmesini yürütür.

### 🎯 Hedef ve Karar Kapısı:
- **Referans Model (Strict Seed 7):** `192/199 = %96.48` Accuracy, `0.9645` Macro-F1.
- **Aday (Motion2 Seed 7):** İki elin de tespit edilemediği karelerde üst gövde ROI ile en az 2 bitişik kare süren el hareketlerinin kurtarıldığı önbellek (1.245 train + 199 val).
- **Başarı Kriteri:** Yeni model $\ge$ 192/199 doğruluk ve değişen videolarda daha az aktif el hatası verirse motion2 yöntemi 100 sınıf ön işleme adayı olarak kabul edilir.
- **Resmi Test Seti:** KESİNLİKLE AÇILMAZ (yalnızca 199 Val videosu değerlendirilir).

### 1. Google Drive Bağlantısı ve Gerekli Kütüphaneler

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Gerekli kütüphaneleri yükle
!pip install -q pyyaml rich opencv-python matplotlib scikit-learn tqdm mediapipe

import os
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print("✅ Google Drive bağlandı, bağımlılıklar hazır.")

### 2. Proje Kodlarının ve Konfigürasyonların Hazırlanması

In [ ]:
import os, shutil

drive_project = "/content/drive/MyDrive/BitirmeProjesi"
local_project = "/content/BitirmeProjesi"

os.makedirs(local_project, exist_ok=True)

# Kod, config ve veri klasörlerini yerel diske kopyala
for folder in ["src", "scripts", "configs", "data"]:
    src_f = os.path.join(drive_project, folder)
    dst_f = os.path.join(local_project, folder)
    if os.path.exists(src_f):
        if os.path.exists(dst_f):
            shutil.rmtree(dst_f)
        shutil.copytree(src_f, dst_f)
        print(f"✅ {folder} klasörü yerel diske kopyalandı.")
    else:
        print(f"⚠️ Uyarı: {src_f} klasörü Drive'da bulunamadı!")

# Sınıf listesini yerel cache'e kopyala
os.makedirs(f"{local_project}/cache", exist_ok=True)
for cf in ["autsl_class_list_10.txt", "class_list_10.txt", "tid_class_list_10.txt"]:
    p = f"{drive_project}/cache/{cf}"
    if os.path.exists(p):
        shutil.copy2(p, f"{local_project}/cache/{cf}")

# Checkpointlerin ve logların kaybolmaması için outputs klasörünü Drive'a bağla
os.makedirs(f"{drive_project}/outputs", exist_ok=True)
if not os.path.exists(f"{local_project}/outputs"):
    os.symlink(f"{drive_project}/outputs", f"{local_project}/outputs")

print("✅ Kodlar ve Drive bağlantısı hazır!")

### 3. Motion2 Önbelleğinin Açılması ve Doğrulanması
*Drive'daki `autsl_10_features_zero_hand_roi_motion2.zip` paketini Colab'ın yerel SSD diskine ultra hızlı (~5-10 saniye) açar.*

In [ ]:
import os, json
from pathlib import Path

cache_name = "autsl_10_features_zero_hand_roi_motion2"
drive_cache_zip = f"{drive_project}/cache/{cache_name}.zip"
drive_cache_dir = f"{drive_project}/cache/{cache_name}"
local_cache_dir = f"{local_project}/cache/{cache_name}"

if not os.path.exists(f"{local_cache_dir}/manifest.json"):
    if os.path.exists(drive_cache_zip):
        print("⚡ Önbellek zip dosyasından yerel SSD'ye açılıyor...")
        !unzip -q -n "$drive_cache_zip" -d "/content/BitirmeProjesi/cache/"
    elif os.path.exists(drive_cache_dir):
        print("📋 Önbellek Drive klasöründen kopyalanıyor...")
        shutil.copytree(drive_cache_dir, local_cache_dir)

# Doğrulama
root = Path(local_cache_dir)
manifest = json.loads((root / 'manifest.json').read_text(encoding='utf-8'))
assert manifest['derivation']['method'] == 'zero_hand_upperbody_roi_motion2'
assert manifest['derivation']['minimum_adjacent_sampled_frames'] == 2
train_count = len(list((root / 'train').glob('*.npz')))
val_count = len(list((root / 'val').glob('*.npz')))
assert train_count == 1245, f"Hata: Train {train_count} (Beklenen: 1245)"
assert val_count == 199, f"Hata: Val {val_count} (Beklenen: 199)"

print(f"✅ Motion2 Önbelleği Tam ve Doğrulandı: {train_count} Train + {val_count} Val dosyası hazır!")

### 4. 🚀 Model Eğitimi (20 Epoch, Seed=7, MobileNetV3-Small Hibrit)
*Eğitim yaklaşık 15-20 dakika sürer. En iyi checkpoint doğrudan Drive'a (`outputs/autsl_10_zero_hand_roi_motion2_seed7/checkpoints/best_model.pt`) kaydedilir.*

In [ ]:
%cd /content/BitirmeProjesi
import os
os.environ["AUTSL_DATA_DIR"] = "/content/drive/MyDrive/BitirmeProjesi/outputs/colab_training_metadata"

!python scripts/train.py \
    --config configs/experiment/autsl_10_zero_hand_roi_motion2_seed7.yaml \
    --cache_dir cache/autsl_10_features_zero_hand_roi_motion2

print("\n✅ Model eğitimi tamamlandı!")

### 5. 📊 Resmi Val Kümesinde Model Değerlendirmesi

In [ ]:
%cd /content/BitirmeProjesi
run_name = "autsl_10_zero_hand_roi_motion2_seed7"
ckpt_p = f"/content/drive/MyDrive/BitirmeProjesi/outputs/{run_name}/checkpoints/best_model.pt"
out_metrics = f"/content/drive/MyDrive/BitirmeProjesi/outputs/{run_name}/val_metrics.json"
out_preds = f"/content/drive/MyDrive/BitirmeProjesi/outputs/{run_name}/val_predictions.json"

!python scripts/evaluate.py \
    --config configs/experiment/autsl_10_zero_hand_roi_motion2_seed7.yaml \
    --checkpoint "$ckpt_p" \
    --cache_dir cache/autsl_10_features_zero_hand_roi_motion2 \
    --split val \
    --output "$out_metrics" \
    --predictions_output "$out_preds"

print("\n✅ Resmi val değerlendirmesi tamamlandı!")

### 6. 🔍 Strict Seed 7 Referansı ile Otomatik Karşılaştırma ve Karar Kapısı

In [ ]:
import json
from pathlib import Path

drive_p = Path("/content/drive/MyDrive/BitirmeProjesi/outputs")
run_name = "autsl_10_zero_hand_roi_motion2_seed7"

baseline_val_p = drive_p / "centerrepair_seed7_val.json"
baseline_pred_p = drive_p / "centerrepair_seed7_predictions.json"
new_val_p = drive_p / run_name / "val_metrics.json"
new_pred_p = drive_p / run_name / "val_predictions.json"

print("=" * 70)
print("📊 MODEL KARŞILAŞTIRMA VE KARAR KAPISI RAPORU")
print("=" * 70)

if new_val_p.exists():
    with open(new_val_p) as f:
        new_val = json.load(f)
    print(f"🎯 YENİ MODEL (Motion2 Seed 7) : Doğruluk = %{new_val.get('accuracy', 0)*100:.2f} | Macro-F1 = {new_val.get('f1_macro', 0):.4f}")

if baseline_val_p.exists():
    with open(baseline_val_p) as f:
        base_val = json.load(f)
    print(f"📌 ESKİ MODEL (Strict Seed 7)   : Doğruluk = %{base_val.get('accuracy', 0)*100:.2f} | Macro-F1 = {base_val.get('f1_macro', 0):.4f}")
    
    diff_acc = (new_val.get('accuracy', 0) - base_val.get('accuracy', 0)) * 100
    diff_f1 = new_val.get('f1_macro', 0) - base_val.get('f1_macro', 0)
    print(f"📈 FARK: Doğruluk: {diff_acc:+.2f}% | Macro-F1: {diff_f1:+.4f}")

# Video bazlı hata değişimi analizi
if baseline_pred_p.exists() and new_pred_p.exists():
    with open(baseline_pred_p) as f:
        b_raw = json.load(f)
        b_list = b_raw.get('predictions', b_raw if isinstance(b_raw, list) else [])
        base_preds = {item['video_id']: item for item in b_list}
    with open(new_pred_p) as f:
        n_raw = json.load(f)
        n_list = n_raw.get('predictions', n_raw if isinstance(n_raw, list) else [])
        new_preds = {item['video_id']: item for item in n_list}
    
    improved = []
    regressed = []
    for vid, item in new_preds.items():
        if vid in base_preds:
            b_pred = base_preds[vid].get('pred_label', base_preds[vid].get('predicted_label'))
            n_pred = item.get('pred_label', item.get('predicted_label'))
            b_correct = (base_preds[vid]['true_label'] == b_pred)
            n_correct = (item['true_label'] == n_pred)
            if not b_correct and n_correct:
                improved.append((vid, item['true_label'], n_pred))
            elif b_correct and not n_correct:
                regressed.append((vid, item['true_label'], n_pred))
    
    print(f"\n🔎 VİDEO DÜZEYİNDE FARKLAR:")
    print(f"   - Düzelen Videolar ({len(improved)} adet): {improved}")
    print(f"   - Gerileyen Videolar ({len(regressed)} adet): {regressed}")

print("=" * 70)
if new_val.get('accuracy', 0) >= base_val.get('accuracy', 0):
    print("🎉 KARAR: Motion2 yöntemi Strict Seed 7 referansını yakaladı veya geçti!")
    print("   100 sınıf ön işlemeye taşınması uygundur.")
else:
    print("⚠️ KARAR: Motion2 yöntemi Strict Seed 7'nin altında kaldı; Strict v1 ile devam edilmesi önerilir.")
print("=" * 70)
